# Комфортная маршрутизация по пресетам

Пайплайн идёт по этапам, каждому этапу соответствует модуль `src/cycle_routing`:

| этап | модуль |
|---|---|
| 1. данные: скачивание и загрузка | `data.py` |
| 2. граф OSM | `graph.py` |
| 3. маппинг наблюдений на рёбра | `matching.py` |
| 4. алгоритм: веса комфорта, повороты, маршруты | `comfort.py`, `routing.py` |
| 5. метрики | `evaluation.py` |
| 6. визуализация | `visualization.py` |

Основной эксперимент — **Konstanz** (реальные потоки STADTRADELN 2024). **Санкт-Петербург** — проверка пресетов на личных GPX.

In [1]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from pyproj import Geod
from shapely.geometry import Point

from cycle_routing import (
    COMFORT_PRESETS,
    GraphConfig,
    build_routes,
    comfort_feature_table,
    compare_route_models,
    composition_table,
    evaluate_corridor_popularity,
    load_clean_personal_gpx,
    load_graph,
    load_stadtradeln,
    match_stadtradeln_to_graph,
    nodes_near_observations,
    route_edge_association,
    route_edges_long,
    sample_od_pairs,
    snap_tracks_to_edges,
    track_edge_usage,
)
from cycle_routing.visualization import ROAD_CLASS_COLORS, SURFACE_CLASS_COLORS, Panel, route_feature_grid

warnings.filterwarnings("ignore", message="GPKG: unrecognized user_version.*")
pd.set_option("display.max_columns", 60)

DATA_DIR = PROJECT_ROOT / "notebooks" / "data"
OSM_DIR = DATA_DIR / "osm"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
ROUTES_CACHE_DIR = OUTPUT_DIR / "routes_cache"  # delete to rebuild routes, e.g. after re-downloading OSM

STAD_URL = (
    "https://offenedaten-konstanz.de/sites/default/files/"
    "Verkehrsmengen%202.0_SR%202024_Konstanz_UTM32_je_Wochentag_gesamt.zip"
)
START_ADDRESS = "Гражданский проспект, 27 к2, Санкт-Петербург, Россия"
END_ADDRESS = "Биржевая линия, 14, Санкт-Петербург, Россия"
SPB_CRS = "EPSG:32636"
N_OD_PAIRS = 20
SEED = 42
SAMPLE_STEP_M = 25.0      # step of points along a route for metrics
MATCH_DISTANCE_M = 25.0   # route point -> STADTRADELN segment / GPX point -> edge
YOUR_STREET_MIN_TRACKS = 2  # an edge counts as "your street" from this many tracks

MODELS = ["shortest", *COMFORT_PRESETS]
LABELS = {"shortest": "кратчайший", **{key: preset["label"] for key, preset in COMFORT_PRESETS.items()}}

d:\PythonProjects\CicleGpx\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Пресеты

Пресет = веса комфорта (`comfort_cost = length × comfort_factor`) + штрафы поворотов в метрах. Кратчайший маршрут считается без того и другого. Пресеты лежат в `COMFORT_PRESETS` (`config.py`).

In [2]:
pd.DataFrame({
    preset["label"]: {
        "описание": preset["description"],
        "primary": preset["config"]["highway"]["primary"],
        "secondary": preset["config"]["highway"]["secondary"],
        "cycleway": preset["config"]["highway"]["cycleway"],
        "велополоса ×": preset["config"]["multipliers"]["dedicated_cycleway"],
        "брусчатка": preset["config"]["surface"]["sett"],
        "гравий": preset["config"]["surface"]["gravel"],
        "поворот налево, м": preset["turns"].left_penalty_m,
    }
    for preset in COMFORT_PRESETS.values()
}).T

,описание,primary,secondary,cycleway,велополоса ×,брусчатка,гравий,"поворот налево, м"
сбалансированный,Исходные веса: умеренный штраф крупных дорог и...,2.1,1.55,0.55,0.65,1.45,1.35,30.0
быстрый,Почти кратчайший путь: тип дороги и покрытие п...,1.15,1.05,0.85,0.9,1.2,1.2,30.0
минимум поворотов,"Длинные прямые участки: веса как у быстрого, п...",1.15,1.05,0.85,0.9,1.2,1.2,300.0
по проспектам,Как в ваших треках по СПб: крупные улицы без ш...,1.0,1.0,0.55,0.5,1.45,1.35,30.0
спокойный,Подальше от машин: штраф улиц с движением и ск...,4.5,3.2,0.5,1.0,1.45,1.35,30.0
велоинфраструктура,"Только велодорожки и велополосы, остальные ули...",2.1,1.55,0.3,0.35,1.45,1.35,30.0
гладкий асфальт,"Шоссейный велосипед: сильный штраф брусчатки, ...",1.6,1.3,0.55,0.65,2.8,3.0,30.0
парки и грунт,"Гревел и прогулка: тропы, парки и грунт в плюс...",3.0,2.2,0.6,0.65,1.45,1.0,30.0


## 1. Konstanz

### 1.1 Данные

STADTRADELN — кампания, в которой участники записывают велопоездки; Konstanz публикует число поездок по каждому сегменту улицы.

In [3]:
observed = load_stadtradeln(
    DATA_DIR / "stadtradeln_konstanz", url=STAD_URL, archive_name="stadtradeln_konstanz_2024_traffic_volumes.zip"
)
popular_threshold = float(observed["number_of_matched_trips"].quantile(0.75))
print(f"{len(observed):,} сегментов; популярный сегмент — от {popular_threshold:.0f} поездок (p75)")

42,291 сегментов; популярный сегмент — от 61 поездок (p75)


### 1.2 Граф

`retain_all=True`: иначе OSMnx оставит только крупнейшую компоненту и часть сегментов не на что будет привязать.

In [4]:
left, bottom, right, top = observed.to_crs("EPSG:4326").total_bounds
G_de = load_graph(
    OSM_DIR, "konstanz", GraphConfig(retain_all=True), observed.crs,
    bbox=(left - 0.005, bottom - 0.005, right + 0.005, top + 0.005),
)
print(f"Граф: {G_de.number_of_nodes():,} узлов, {G_de.number_of_edges():,} рёбер")

Граф: 41,254 узлов, 100,734 рёбер


### 1.3 Маппинг поездок на рёбра

Сегмент ищется по `osm_way_id` с проверкой геометрии, иначе ближайшее ребро в пределах 15 м. Таблица признаков — то, из чего складывается `comfort_factor`; ниже по одному ребру каждого типа.

In [5]:
G_de, match_metrics = match_stadtradeln_to_graph(G_de, observed)
print(
    f"Привязано {match_metrics['stad_match_rate']:.1%} сегментов, расхождение p95 = "
    f"{match_metrics['p95_match_distance_m']:.1f} м; поездки есть у {match_metrics['osm_edge_match_rate']:.0%} рёбер"
)
de_features = comfort_feature_table(G_de)
de_features.groupby("road_class", sort=False).head(1)

Привязано 94.8% сегментов, расхождение p95 = 7.0 м; поездки есть у 22% рёбер


,,,highway,road_class,surface,surface_class,has_cycleway,footway_without_bicycle,maxspeed_kmh,lanes,unlit,length_m,highway_factor,surface_factor,cycleway_mult,footway_mult,maxspeed_mult,lanes_mult,lit_mult,comfort_factor,comfort_cost,stad_trips,geometry
u,v,key,,,,,,,,,,,,,,,,,,,,,
2104826,300373377,0,trunk,крупная дорога,asphalt,асфальт/бетон,False,False,120.0,2.0,False,15.347149,3.50,1.00,1.00,1.0,1.5,1.0,1.0,5.2500,80.572534,0.0,"LINESTRING (510904.57 5293356.095, 510918.684 ..."
2104833,84683569,0,tertiary,улица,asphalt,асфальт/бетон,False,False,50.0,3.0,False,135.122582,1.25,1.00,1.00,1.0,1.0,1.2,1.0,1.5000,202.683873,0.0,"LINESTRING (511047.134 5293259.105, 511045.993..."
2104851,14057068875,0,track,тропа/грунтовка,NaN,не указано,False,False,NaN,NaN,False,180.193824,1.20,1.20,1.00,1.0,1.0,1.0,1.0,1.4400,259.479107,0.0,"LINESTRING (511456.208 5291829.79, 511462.437 ..."
2260605658,2459652672,0,path,велодорожка,asphalt;paving_stones,асфальт/бетон,False,False,NaN,NaN,False,183.419135,0.85,1.12,1.00,1.0,1.0,1.0,1.0,0.9520,174.615017,0.0,"LINESTRING (511307.287 5291692.726, 511307.019..."
2105103,7861954235,0,secondary,крупная + велополоса,asphalt,асфальт/бетон,True,False,50.0,1.0,False,23.531263,1.55,1.00,0.65,1.0,1.0,1.0,1.0,1.0075,23.707748,0.0,"LINESTRING (534009.168 5278344.217, 533985.796..."
33431244,264397444,0,pedestrian;service,тротуар/пешеходная,paving_stones,плитка,False,False,NaN,NaN,False,108.817546,1.10,1.12,1.00,1.0,1.0,1.0,1.0,1.2320,134.063217,0.0,"LINESTRING (520204.855 5282278.062, 520195.95 ..."


### 1.4 Маршруты

Случайные пары старт–финиш 1.5–10 км внутри зоны с данными; для каждой — кратчайший маршрут и маршрут каждого пресета.

In [6]:
de_pairs = sample_od_pairs(G_de, nodes_near_observations(G_de, observed, MATCH_DISTANCE_M), N_OD_PAIRS, SEED)
de_routes = build_routes(G_de, de_pairs, COMFORT_PRESETS, cache_dir=ROUTES_CACHE_DIR)

Routes by preset: 100%|██████████| 8/8 [00:00<00:00, 10.82it/s]


### 1.5 Метрики

- **объезд** — насколько длиннее кратчайшего; **поворотов** — левые, правые и развороты;
- **поездок × кратчайший** — во сколько раз больше поездок STADTRADELN вдоль маршрута (медиана по парам);
- **популярные улицы** — доля маршрута по сегментам выше p75;
- **связь с популярностью** — корреляция «ребро выбрано» с `log(1 + поездки)` среди улиц в 300 м от маршрутов.

In [7]:
de_metrics = evaluate_corridor_popularity(
    de_routes, observed, sample_step_m=SAMPLE_STEP_M, match_distance_m=MATCH_DISTANCE_M,
    popular_threshold=popular_threshold,
)
comparison = compare_route_models(de_metrics)
gain = comparison.groupby("model")["popularity_gain"]
by_model = de_metrics.groupby("algorithm")
association = route_edge_association(de_routes, de_features, "stad_trips")
de_summary = pd.DataFrame({
    "объезд, %": comparison.groupby("model")["detour_pct"].median(),
    "поворотов": by_model["turns"].median(),
    "поездок × кратчайший": np.exp(gain.median()),
    "популярные улицы, %": 100 * by_model["popular_edge_share"].median(),
    "популярнее кратчайшего": gain.apply(lambda g: f"{(g > 0).sum()}/{len(g)}"),
    "связь с популярностью": association,
}).reindex(MODELS).fillna({"объезд, %": 0.0, "поездок × кратчайший": 1.0}).rename(index=LABELS)
de_summary.style.format({
    "объезд, %": "{:.1f}", "поворотов": "{:.0f}", "поездок × кратчайший": "{:.2f}",
    "популярные улицы, %": "{:.0f}", "связь с популярностью": "{:.2f}",
}, na_rep="—")

STAD matching: 100%|██████████| 180/180 [00:02<00:00, 82.11it/s]


,"объезд, %",поворотов,поездок × кратчайший,"популярные улицы, %",популярнее кратчайшего,связь с популярностью
кратчайший,0.0,14,1.00,59,—,0.16
сбалансированный,5.9,10,2.06,85,18/20,0.26
быстрый,3.2,12,1.31,74,16/20,0.23
минимум поворотов,10.7,6,1.41,71,16/20,0.22
по проспектам,10.0,10,2.30,88,18/20,0.26
спокойный,10.5,14,2.34,82,18/20,0.26
велоинфраструктура,19.1,12,2.40,86,17/20,0.27
гладкий асфальт,12.3,13,1.79,78,15/20,0.23
парки и грунт,6.5,12,1.32,67,15/20,0.18


### 1.6 Визуализация

Пара, где пресеты дали больше всего разных маршрутов. Обводка — пресет, цвет линии — признак ребра; меню слоёв справа сверху включает пресеты и меняет подложку сразу на всех картах.

In [8]:
distinct = de_routes.groupby("od_id")["edge_route"].agg(lambda r: len({tuple(map(tuple, route)) for route in r}))
de_example_od = distinct.idxmax()
_, origin, destination = next(pair for pair in de_pairs if pair[0] == de_example_od)
ends = gpd.GeoSeries(
    [Point(G_de.nodes[n]["x"], G_de.nodes[n]["y"]) for n in (origin, destination)], crs=G_de.graph["crs"]
).to_crs("EPSG:4326")
de_map = route_feature_grid(
    G_de,
    de_routes.query("od_id == @de_example_od"),
    de_features,
    [
        Panel("Популярность (поездки STADTRADELN)", "stad_trips", log=True, higher_is_better=True),
        Panel("Тип дороги", "road_class", categories=ROAD_CLASS_COLORS),
        Panel("Покрытие", "surface_class", categories=SURFACE_CLASS_COLORS),
        Panel("Коэффициент комфорта (сбалансированный)", "comfort_factor", low_label="комфортно", high_label="некомфортно"),
    ],
    (ends.y[0], ends.x[0]),
    (ends.y[1], ends.x[1]),
)
print(f"OD-пара {de_example_od}")
de_map

OD-пара DE-02


## 2. Санкт-Петербург

### 2.1 Данные

Личные GPX-треки нужны только как аналог популярности: по каким улицам вы ездите. Маршрут строится один раз — между адресами.

In [9]:
personal = load_clean_personal_gpx(PROJECT_ROOT / "cicle_gpx").to_crs(SPB_CRS)
start = ox.geocoder.geocode_to_gdf(START_ADDRESS, which_result=1).iloc[0]  # plain geocode lands off the building
start_latlon = (float(start["lat"]), float(start["lon"]))
end_latlon = ox.geocoder.geocode(END_ADDRESS)
print(f"{len(personal)} треков, медиана {personal.length.median() / 1000:.1f} км")

19 треков, медиана 10.8 км


### 2.2 Граф

In [10]:
lats = [start_latlon[0], end_latlon[0], *personal.to_crs("EPSG:4326").total_bounds[[1, 3]]]
lons = [start_latlon[1], end_latlon[1], *personal.to_crs("EPSG:4326").total_bounds[[0, 2]]]
center = ((min(lats) + max(lats)) / 2, (min(lons) + max(lons)) / 2)
radius_m = max(abs(Geod(ellps="WGS84").inv(center[1], center[0], lon, lat)[2]) for lat, lon in zip(lats, lons)) + 1200
G_spb = load_graph(OSM_DIR, "saint_petersburg", GraphConfig(), SPB_CRS, center=center, dist=radius_m, largest_component=True)
print(f"Граф: {G_spb.number_of_nodes():,} узлов, {G_spb.number_of_edges():,} рёбер")

Граф: 52,457 узлов, 120,323 рёбер


### 2.3 Маппинг треков на рёбра

Точки трека через 25 м привязываются к ближайшему ребру; `gpx_tracks` — сколько треков прошло по ребру.

In [11]:
spb_features = comfort_feature_table(G_spb)
spb_features["gpx_tracks"] = track_edge_usage(
    snap_tracks_to_edges(G_spb, personal, step_m=SAMPLE_STEP_M, max_distance_m=MATCH_DISTANCE_M), spb_features
)
print(f"Ваши треки проходят по {int((spb_features['gpx_tracks'] > 0).sum()):,} рёбрам")

Ваши треки проходят по 2,394 рёбрам


### 2.4 Маршрут

Кратчайший и по одному маршруту на пресет от Гражданского проспекта до Биржевой линии.

In [12]:
address_xy = gpd.GeoSeries(
    [Point(start_latlon[1], start_latlon[0]), Point(end_latlon[1], end_latlon[0])], crs="EPSG:4326"
).to_crs(SPB_CRS)
origin, destination = ox.distance.nearest_nodes(G_spb, X=address_xy.x.to_numpy(), Y=address_xy.y.to_numpy())
spb_routes = build_routes(G_spb, [("address", int(origin), int(destination))], COMFORT_PRESETS, cache_dir=ROUTES_CACHE_DIR)

Routes by preset: 100%|██████████| 8/8 [00:00<00:00,  8.79it/s]


### 2.5 Метрики

- **по вашим улицам** — доля длины маршрута по рёбрам, где проехали минимум `YOUR_STREET_MIN_TRACKS` ваших трека (аналог «популярных улиц» в Konstanz);
- **объезд** — к кратчайшему; **поворотов** — левые, правые, развороты; **крупные улицы** — primary/secondary.

In [13]:
edges = route_edges_long(spb_routes, spb_features)
edges["tracks"] = spb_features["gpx_tracks"].reindex(pd.MultiIndex.from_frame(edges[["u", "v", "key"]])).to_numpy()
major = composition_table(edges, spb_features, "road_class").reindex(
    columns=["крупная дорога", "крупная + велополоса"], fill_value=0.0
)
routes_by_model = spb_routes.set_index("algorithm")
spb_summary = pd.DataFrame({
    "длина, км": routes_by_model["route_length_m"] / 1000,
    "объезд, %": 100 * (routes_by_model["route_length_m"] / routes_by_model.loc["shortest", "route_length_m"] - 1),
    "поворотов": routes_by_model["turns"],
    "по вашим улицам, %": edges.groupby("group").apply(
        lambda g: 100 * g["weight"][g["tracks"] >= YOUR_STREET_MIN_TRACKS].sum() / g["weight"].sum()
    ),
    "крупные улицы, %": major.sum(axis=1),
}).reindex(MODELS).rename(index=LABELS)
spb_summary.style.format({
    "длина, км": "{:.1f}", "объезд, %": "{:.1f}", "поворотов": "{:.0f}", "по вашим улицам, %": "{:.0f}", "крупные улицы, %": "{:.0f}",
})

,F1,лучше кратчайшего,"объезд, %",поворотов,"крупные улицы, %"
кратчайший,0.47,0/19,0.0,14,82
сбалансированный,0.11,1/19,11.3,24,37
быстрый,0.55,10/19,0.6,6,85
минимум поворотов,0.67,14/19,0.8,4,92
по проспектам,0.59,12/19,0.6,6,91
спокойный,0.05,1/19,34.0,55,11
велоинфраструктура,0.50,6/19,6.7,13,71
гладкий асфальт,0.41,5/19,5.3,13,71
парки и грунт,0.08,1/19,20.1,43,15


### 2.6 Визуализация

In [14]:
spb_map = route_feature_grid(
    G_spb,
    spb_routes,
    spb_features,
    [
        Panel(f"Ваши треки по ребру (из {len(personal)})", "gpx_tracks", low_label="0", high_label="много", higher_is_better=True),
        Panel("Тип дороги", "road_class", categories=ROAD_CLASS_COLORS),
        Panel("Покрытие", "surface_class", categories=SURFACE_CLASS_COLORS),
        Panel("Коэффициент комфорта (сбалансированный)", "comfort_factor", low_label="комфортно", high_label="некомфортно"),
    ],
    start_latlon,
    end_latlon,
)
spb_map

## 3. Сохранение

In [15]:
de_summary.to_csv(OUTPUT_DIR / "konstanz_presets.csv")
spb_summary.to_csv(OUTPUT_DIR / "spb_presets.csv")
de_map.save(str(OUTPUT_DIR / "konstanz_presets_map.html"))
spb_map.save(str(OUTPUT_DIR / "spb_presets_map.html"))
print(f"Сохранено в {OUTPUT_DIR}")

Сохранено в d:\PythonProjects\CicleGpx\outputs
